In [ ]:
# Import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px

In [ ]:
# Load the datasets
purchase_behaviour = pd.read_csv("QVI_purchase_behaviour.csv")
transaction_data = pd.read_excel("QVI_transaction_data.xlsx")

In [ ]:
# View the first few rows of purchase_behaviour data
purchase_behaviour.head()

In [ ]:
# Check the formats of the data received
purchase_behaviour.info()

In [ ]:
# View the first few rows of transaction_data
transaction_data.head()

In [ ]:
# Check the formats of the data received
transaction_data.info()

In [ ]:
# We can see that the date column is in an integer format. Let's change this to a date format.
transaction_data['DATE'] = pd.to_datetime(transaction_data['DATE'], origin='1899-12-30', unit='D')

In [ ]:
# Examine product names
transaction_data['PROD_NAME'].describe

In [ ]:
# Split product names into words
product_words = (
    transaction_data['PROD_NAME']
    .str.split()
    .explode()
)

# Remove numbers and symbols
product_words = product_words[
    product_words.str.contains(r'^[A-Za-z]+$', regex=True)
]

# Count frequency
word_counts = product_words.value_counts()
print(word_counts.head(20))

In [ ]:
# There are salsa products in the dataset but we are only interested in the chips category, so let's remove these.
transaction_data = transaction_data[~transaction_data['PROD_NAME'].str.lower().str.contains("salsa")]

transaction_data.describe()

In [ ]:
"""
There are no nulls in the columns but product quantity appears to have an outlier 
which we should investigate further. Let's investigate further the case where 200
packets of chips are bought in one transaction.
"""
transaction_data[transaction_data['PROD_QTY'] == 200]


# Since this customer has only two transactions and is not a regular retail buyer we would remove the loyalty card number from further analysis
outlier_customer = transaction_data.loc[transaction_data['PROD_QTY'] == 200, 'LYLTY_CARD_NBR'].unique()

transaction_data = transaction_data[~transaction_data['LYLTY_CARD_NBR'].isin(outlier_customer)]

In [ ]:
# Aggregate data
transactions_by_day = (transaction_data.groupby('DATE')
    .size()
    .reset_index(name='TRANSACTION_COUNT')
)

"""
There's only 364 rows, meaning only 364 dates which indicates a missing date. Let's
create a sequence of dates from 1 Jul 2018 to 30 Jun 2019 and use this to create a
chart of number of transactions over time to find the missing date.
"""

# Create full date range
full_dates = pd.date_range(
    start=transactions_by_day['DATE'].min(),
    end=transactions_by_day['DATE'].max()
)

full_df = pd.DataFrame({'DATE': full_dates})

# Merge
transactions_full = full_df.merge(
    transactions_by_day,
    on='DATE',
    how='left'
).fillna(0)

# Interactive line chart
fig = px.line(
    transactions_full,
    x='DATE',
    y='TRANSACTION_COUNT',
    title='Transactions Over Time (Including Missing Dates)'
)
# Add Range Slider + Zoom (Critical for Time Analysis)
fig.update_layout(
    xaxis=dict(rangeslider=dict(visible=True), type="date")
)

fig.show()

In [ ]:
# Zoom into December
december_data = transactions_full[
    (transactions_full['DATE'] >= '2018-12-01') &
    (transactions_full['DATE'] <= '2018-12-31')
]

fig = px.line(
    december_data,
    x='DATE',
    y='TRANSACTION_COUNT',
    title='December Transactions (Holiday Trend Analysis)'
)
# Add Insight Annotations
fig.add_annotation(
    x='2018-12-25',
    y=0,
    text='Christmas Day (Store Closed)',
    showarrow=True,
    arrowhead=2
)

fig.show()

"""
We can see that the increase in sales occurs in the lead-up to Christmas and that
there are zero sales on Christmas day itself. This is due to shops being closed on
Christmas day.
Now that we are satisfied that the data no longer has outliers, we can move on to
creating other features.
"""

In [ ]:
# Pack sizes and Brand Analysis
transaction_data['PACK_SIZE'] = transaction_data['PROD_NAME'].str.extract(r'(\d+)').astype(float)
transaction_data['BRAND'] = transaction_data['PROD_NAME'].str.split().str[0]
transaction_data['BRAND'] = transaction_data['BRAND'].replace({'Red': 'RRD'})

In [ ]:
# Since we have performed the necessary clean ups of our datasets; it's time we merge them
data = pd.merge(transaction_data, purchase_behaviour, on='LYLTY_CARD_NBR', how='left')

data.isnull().sum()
data.to_csv('data.csv', index=False)

In [ ]:
# Data analysis on customer segments
sales = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TOT_SALES'].sum().reset_index()

plt.figure(figsize=(10,6))
sns.barplot(data=sales, x='LIFESTAGE', y='TOT_SALES', hue='PREMIUM_CUSTOMER')
plt.xticks(rotation=90)
plt.title("Total Sales by Customer Segment")
plt.show()

In [ ]:
#Number of Customers
customers = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['LYLTY_CARD_NBR'].nunique().reset_index()

#Avg Units per Customer
units = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['PROD_QTY'].mean().reset_index()

#Avg Price per Unit
data['PRICE_PER_UNIT'] = data['TOT_SALES'] / data['PROD_QTY']

price = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['PRICE_PER_UNIT'].mean().reset_index()


In [ ]:
#T-Test (Statistical Test)
from scipy.stats import ttest_ind

mainstream = data[
    (data['PREMIUM_CUSTOMER'] == 'Mainstream')
]['PRICE_PER_UNIT']

others = data[
    data['PREMIUM_CUSTOMER'].isin(['Budget', 'Premium'])
]['PRICE_PER_UNIT']

t_stat, p_value = ttest_ind(mainstream, others, equal_var=False)

print("P-value:", p_value)

In [ ]:
#Brand Preference
segment = data[
    (data['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') &
    (data['PREMIUM_CUSTOMER'] == 'Mainstream')
]

brand_pref = segment['BRAND'].value_counts(normalize=True)

#Pack Size Preference
pack_pref = segment['PACK_SIZE'].value_counts(normalize=True)